# Fine-tuning the parser: Qwen3.5-0.8B with LoRA

Trains the small model on the examples of `daily.datagen` (labelled by construction) and
exports a `Q4_0` GGUF for `llama-server`. The examples use the prompt shape the bot sends:
changing it in `daily/agent/prompts.py` means training again.

Runtime: **GPU** (Runtime → Change runtime type → T4 is enough). About 30-45 minutes on a T4.
The results go to Google Drive, in `daily-finetune/` (the first cell asks for access).

In [ ]:
# Settings
REPO = "https://github.com/fpetroli/daily.git"
BRANCH = "dev"
SINGLE, DIALOGUES, SEED = 500, 250, 1   # the same values give the same examples
BASE_MODEL = "unsloth/Qwen3.5-0.8B"
EPOCHS = 2
OUT = "/content/drive/MyDrive/daily-finetune"
!nvidia-smi --query-gpu=name,memory.total --format=csv

from google.colab import drive
drive.mount("/content/drive")  # asked now, so the run does not stop at the end
import os
os.makedirs(OUT, exist_ok=True)

## 1. The examples

The generator runs the real agent, so it needs the repository and its dependencies: they go
in a separate virtual environment, away from the training libraries.

In [ ]:
import subprocess

def sh(command):
    # stops the notebook if the command fails: a `!` line does not
    print(f"$ {command}")
    subprocess.run(command, shell=True, check=True)

sh("pip install -q uv")
sh(f"rm -rf /content/daily && git clone -q --depth 1 -b {BRANCH} {REPO} /content/daily")
sh("uv venv -q --python 3.12 --clear /content/daily-venv")  # python3 -m venv lacks ensurepip on Colab
sh("uv pip install -q --python /content/daily-venv/bin/python -e /content/daily pyyaml")
sh(
    "cd /content/daily && /content/daily-venv/bin/python -m daily.datagen "
    f"--count {SINGLE} --dialogues {DIALOGUES} --seed {SEED} --out /content/train.jsonl"
)

## 2. The training libraries (from the official Unsloth Qwen3.5 notebook)

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

## 3. The model

16-bit LoRA: Unsloth advises against 4-bit for Qwen3.5 (larger quantization error).

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_LENGTH = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_LENGTH,
)

## 4. Prompt and answer, exactly as `llama-server` writes them

The server runs with `--reasoning off`, which renders the template with
`enable_thinking=False`. The prompt is that rendering; the loss is only on the answer (the
compact tool call and the end-of-turn token).

In [ ]:
import json, random
from datasets import Dataset

text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)  # a processor wraps the tokenizer
END = "<|im_end|>"

def to_pair(row):
    *chat, answer = row["messages"]
    prompt = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    return {"prompt": prompt, "completion": answer["content"] + END, "source": row["source"]}

rows = [json.loads(line) for line in open("/content/train.jsonl", encoding="utf-8")]
pairs = [to_pair(row) for row in rows]
random.Random(0).shuffle(pairs)
lengths = [len(text_tokenizer(p["prompt"] + p["completion"]).input_ids) for p in pairs]
print(f"{len(pairs)} examples, tokens: max {max(lengths)}, mean {sum(lengths) // len(lengths)}")
assert max(lengths) <= MAX_LENGTH
print(pairs[0]["prompt"][-700:])
print("ANSWER:", pairs[0]["completion"])

held = max(30, len(pairs) // 20)
train_set = Dataset.from_list(pairs[held:])
check_set = pairs[:held]

## 5. Training

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=text_tokenizer,
    train_dataset=train_set,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS,
        warmup_steps=10,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.001,
        logging_steps=5,
        max_length=MAX_LENGTH,
        completion_only_loss=True,
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f} minutes")

## 6. A quick check on the examples kept aside

Not the eval: the real measure is `daily.evals` on the 216 cases, with the GGUF. This only
tells if the training went wrong (it should be close to 100%).

In [ ]:
FastLanguageModel.for_inference(model)
right = 0
for pair in check_set:
    inputs = text_tokenizer(pair["prompt"], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
    said = text_tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    expected = pair["completion"].removesuffix(END)
    try:
        ok = json.loads(said) == json.loads(expected)
    except json.JSONDecodeError:
        ok = False
    right += ok
    if not ok:
        print(f"[{pair['source']}] expected {expected}\n{' ' * (len(pair['source']) + 3)}got      {said}")
print(f"{right}/{len(check_set)} exact")

## 7. Saving to Google Drive

The LoRA adapter (small, to train again from here) and the `Q4_0` GGUF for `llama-server`,
the same quantization as the base model measured before.

In [ ]:
model.save_pretrained(f"{OUT}/lora")
tokenizer.save_pretrained(f"{OUT}/lora")
model.save_pretrained_gguf("/content/gguf", tokenizer, quantization_method="q4_0")
# Unsloth writes into "/content/gguf_gguf", next to a vision projector (mmproj) we don't need
import glob, shutil
ggufs = glob.glob("/content/gguf*/*Q4_0.gguf")
assert ggufs, "no Q4_0 GGUF was written"
for path in ggufs:
    shutil.copy(path, OUT)
!cp /content/train.jsonl "$OUT"/
!ls -la "$OUT"